In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from pathlib import Path
load_dotenv()

True

In [2]:
def call_llm(prompt):
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [13]:
DATA = Path('data')

standards = (DATA / "team_standards.md").read_text(encoding="utf-8")
payment_code = (DATA / "payment_utils.py").read_text(encoding="utf-8")
auth_code = (DATA / "auth.py").read_text(encoding="utf-8")


In [42]:
# print(standards)

prompt = f"""Review this Python code against our team standards.
Give 2-3 issues with line numbers. Do NOT claim you posted a PR comment.

STANDARDS:
{standards}

FILE : payment.py

{auth_code}

"""



In [43]:
print(call_llm(prompt))


Here are a few issues with the provided code based on the team standards:

1. **Line 2-3:** The `login` function directly compares the username and password without any validation. According to standard #3, all user inputs must be validated before database queries. The function should include checks to ensure that input is in the expected format and not just hard-coded values.

2. **Line 6:** The `generate_token` function concatenates the user ID with a timestamp to create a token. However, this does not comply with standard #4, which states that auth tokens must use JWT with an expiry. The implementation should use a JWT library to generate a proper token that includes an expiry.

3. **Line 3 (and other relevant database operations, if present):** The code does not include any database queries; if the complete code had DB operations, they must be parameterized to adhere to standard #1. Ensure that any database interactions use SQLAlchemy's parameterized queries to prevent SQL injectio

In [ ]:
import requests

In [3]:
## tools
pr = {
     "number": 247,
    "comments": [],
    "status": "PENDING"

}

def read_file(filename):
    return (DATA/filename).read_text(encoding="utf-8")



def post_review_comment(filename,comment):
    pr["comments"].append({"file": filename, "comment": comment})
    return "Comment saved"


def set_pr_status(status):
    if status not in ["PASS", "BLOCKED"]:
        return "ERROR: status must be PASS or BLOCKED"
    pr["status"] = status
    return "PR status updated"

def search(topic):
    url = f"https://api.example.com/search?q={topic}"
    data = requests.get(url).json()
    return data


In [4]:
tools = [
  {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read one sample file to inspect its contents",
            "parameters": {
                "type": "object",
                "properties": {"filename": {"type": "string"}},
                "required": ["filename"]
            }
        }
    },
     {
        "type": "function",
        "function": {
            "name": "post_review_comment",
            "description": "Save a simulated PR review comment for a file",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {"type": "string"},
                    "comment": {"type": "string"}
                },
                "required": ["filename", "comment"]
            }
        }
    },

     {
        "type": "function",
        "function": {
            "name": "set_pr_status",
            "description": "Set simulated PR status after reviewing all files",
            "parameters": {
                "type": "object",
                "properties": {"status": {"type": "string", "enum": ["PASS", "BLOCKED"]}},
                "required": ["status"]
            }
        }
    },
    

]

In [5]:
demo_messages = [
    {"role": "user", "content": "Please read auth.py before reviewing it."}


]

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=demo_messages,
    tools=tools
)


response.choices[0].message.content


In [6]:
print(response)

ChatCompletion(id='chatcmpl-Ef2uGil64TLCpuV3UOdhh6Zd74iLM', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_5TZBREERaGjZaWvflaqx6JhO', function=Function(arguments='{"filename":"auth.py"}', name='read_file'), type='function')]))], created=1793451328, model='gpt-4o-mini-2024-07-18', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_3a7e128613', usage=CompletionUsage(completion_tokens=15, prompt_tokens=114, total_tokens=129, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))


In [8]:
first_message = response.choices[0].message
first_message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_5TZBREERaGjZaWvflaqx6JhO', function=Function(arguments='{"filename":"auth.py"}', name='read_file'), type='function')])

In [79]:
# first_message.tool_calls[0].function.name

In [80]:
# first_message.tool_calls[0].function.name
# first_message.tool_calls[0].function.arguments

In [9]:
call = first_message.tool_calls[0]
call

ChatCompletionMessageFunctionToolCall(id='call_5TZBREERaGjZaWvflaqx6JhO', function=Function(arguments='{"filename":"auth.py"}', name='read_file'), type='function')

In [10]:
import json

In [11]:
arguments = json.loads(first_message.tool_calls[0].function.arguments)
arguments

{'filename': 'auth.py'}

In [15]:
file_text = read_file(arguments['filename'])
file_text


'def login(username, password):\n    if username == "admin" and password == "admin123":\n        return True\n    return False\n\n\ndef generate_token(user_id):\n    import time\n    return f"{user_id}_{time.time()}"'

In [16]:
demo_messages.append(first_message)

In [17]:
demo_messages

[{'role': 'user', 'content': 'Please read auth.py before reviewing it.'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_5TZBREERaGjZaWvflaqx6JhO', function=Function(arguments='{"filename":"auth.py"}', name='read_file'), type='function')])]

In [18]:
demo_messages.append({
     "role": "tool",
     "tool_call_id": call.id,
     "content" : file_text

})

In [19]:
response_se = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=demo_messages,
    tools=tools
)

In [22]:
print(response_se.choices[0].message.content)

The `auth.py` file contains two functions: `login` and `generate_token`.

1. **`login(username, password)`**: This function checks if the provided username and password match the hard-coded values ("admin" and "admin123"). If both match, it returns `True`; otherwise, it returns `False`.

2. **`generate_token(user_id)`**: This function generates a token by concatenating the `user_id` with the current time (in seconds since the epoch). It uses Python's `time` module to get the current time.

### Review Comments
1. **Security Concern**: Hardcoding credentials (username and password) is a significant security risk. It is recommended to use a secure method to authenticate users, such as checking against a database and hashing passwords.
   
2. **Token Generation**: The `generate_token` function creates a simple token using the current time. This could lead to potential security issues (e.g., token collisions) as the same token can be generated if called multiple times within the same second

In [23]:
demo_messages

[{'role': 'user', 'content': 'Please read auth.py before reviewing it.'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_5TZBREERaGjZaWvflaqx6JhO', function=Function(arguments='{"filename":"auth.py"}', name='read_file'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_5TZBREERaGjZaWvflaqx6JhO',
  'content': 'def login(username, password):\n    if username == "admin" and password == "admin123":\n        return True\n    return False\n\n\ndef generate_token(user_id):\n    import time\n    return f"{user_id}_{time.time()}"'}]

## Build the full agent ( single agent) using python

In [35]:
functions = {
    "read_file": read_file,
    "post_review_comment": post_review_comment,
    "set_pr_status": set_pr_status
}



def run_agent(question, max_steps=10):

    # step 1 prepare the messages
    messages = [
        {
            "role": "system",
            "content": (
                "You are a code review agent. "
                "First read team_standards.md. "
                "Then review payment_utils.py, auth.py, and order.py. "
                "Post comments for confirmed important issues. "
                "Set PR status to BLOCKED if important violations exist, "
                "otherwise PASS. Finally, summarize the review."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    step = 0

    while step < max_steps:
        step += 1
        print(f"\n===== Agent Step {step} =====")

        # THINK : Ask LLM what to do next
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            tool_choice='auto'
        )

        message = response.choices[0].message
        # Remember the LLM's response
        messages.append(message)

        # STOP : NO more tools call
        if not message.tool_calls:
            print("agent finished")
            return message.content
        
        for call in  message.tool_calls:
            tool_name = call.function.name
            print(f"Calling tool: {tool_name}")

            try:
                arguments = json.loads(call.function.arguments)
                if tool_name not in functions:
                    raise ValueError(f"Unknown tool: {tool_name}")

                python_function = functions[tool_name]
                result = python_function(**arguments)

            except Exception as e:
                result = f"Tool error: {str(e)}"

            messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": str(result)
                })
            
    return "Agent stopped : Maximum steps reached"



In [37]:
pr = {
     "number": 247,
    "comments": [],
    "status": "PENDING"

}

In [41]:
pr["comments"] = []
pr["status"] = "PENDING"

answer = run_agent("Review PR #247 using our team coding standards.")



===== Agent Step 1 =====
Calling tool: read_file
Calling tool: read_file
Calling tool: read_file

===== Agent Step 2 =====
Calling tool: post_review_comment
Calling tool: post_review_comment
Calling tool: post_review_comment
Calling tool: post_review_comment
Calling tool: post_review_comment

===== Agent Step 3 =====
Calling tool: set_pr_status

===== Agent Step 4 =====
agent finished


In [42]:
print(answer)

### PR Review Summary

The review of PR #247 revealed several important issues related to code security and best practices:

1. **payment_utils.py**:
   - **SQL Injection Risk**: The use of string interpolation in the SQL query could lead to SQL injection vulnerabilities. It is recommended to use parameterized queries.
   - **Sensitive Information Exposure**: In the `process_payment` function, printing the card number poses a security risk. Logging should be considered instead of printing sensitive data.

2. **auth.py**:
   - **Hardcoded Credentials**: The `login` function contains hardcoded credentials, which is a security risk. It's better to use environment variables or secure storage for authentication credentials.
   - **Token Generation Security**: The `generate_token` function does not utilize secure token generation practices. It's advisable to use a dedicated library for secure token generation.

3. **order.py**:
   - **Empty Placeholder**: The file appears to be an empty plac

In [43]:
pr

{'number': 247,
 'comments': [{'file': 'payment_utils.py',
   'comment': 'Important: Using string interpolation in SQL query can lead to SQL injection vulnerabilities. Use parameterized queries instead.'},
  {'file': 'payment_utils.py',
   'comment': 'Important: The function `process_payment` should not print sensitive information such as card number. Consider logging instead.'},
  {'file': 'auth.py',
   'comment': 'Important: Hardcoded credentials in the login function (username and password) are a security risk. Consider using environment variables or a secure storage method for authentication.'},
  {'file': 'auth.py',
   'comment': "Important: The `generate_token` function doesn't follow secure token generation practices. Use a library that guarantees a more secure token."},
  {'file': 'order.py',
   'comment': "Important: The file seems to be an empty placeholder ('# tetsing code'). It should contain implementation or be removed."}],
 'status': 'BLOCKED'}